# Maltese Gear Cube — DeepRL Training on Colab A100

## How to use this notebook

**Before your first run:**
1. Upload all your `.py` files and `test_dataset.pt` to `MyDrive/MalteseGearCube/code/` on Google Drive.
2. In Colab: `Runtime → Change runtime type → A100 GPU`.
3. Run all cells top to bottom.

**If your session disconnects and you reconnect:**
1. Just run all cells top to bottom again — the notebook will detect your checkpoint and resume automatically.

**Cell order (run sequentially):**
- Cell 1 — Verify A100 GPU
- Cell 2 — Mount Google Drive
- Cell 3 — Configure paths
- Cell 4 — Copy code from Drive to local Colab storage
- Cell 5 — Resume check: copy existing checkpoint from Drive to local
- Cell 6 — Start background Drive sync (every 5 min)
- Cell 7 — Start / resume training
- Cell 8 — Manual final sync (run this after training completes)

---
## Cell 1 — Verify A100 GPU

In [ ]:
import subprocess
import torch

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected!\n"
        "Go to Runtime → Change runtime type → GPU → A100, then re-run."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")

if vram_gb < 35:
    print()
    print(f"WARNING: Got {vram_gb:.1f} GB VRAM. Hyperparameters are tuned for A100 40 GB.")
    print("Training will still work but may be slower or OOM on smaller GPUs.")
else:
    print()
    print("A100 40 GB confirmed — good to go!")


---
## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
print("Drive mounted at /content/drive")


---
## Cell 3 — Configure Paths

In [ ]:
import os

# ── Drive paths (persistent across sessions) ──────────────────────────────────
DRIVE_BASE  = '/content/drive/MyDrive/MalteseGearCube'
DRIVE_CODE  = f'{DRIVE_BASE}/code'          # put your .py files here
DRIVE_SAVE  = f'{DRIVE_BASE}/saved_models'  # checkpoints backed up here

# ── Local Colab paths (fast NVMe, lost on disconnect) ─────────────────────────
LOCAL_BASE  = '/content/MalteseGearCube'
LOCAL_CODE  = f'{LOCAL_BASE}/code'
LOCAL_SAVE  = f'{LOCAL_BASE}/saved_models'

# Create all directories
for d in [
    DRIVE_BASE, DRIVE_CODE, DRIVE_SAVE,
    f'{DRIVE_SAVE}/current', f'{DRIVE_SAVE}/target',
    LOCAL_BASE, LOCAL_CODE, LOCAL_SAVE,
    f'{LOCAL_SAVE}/current', f'{LOCAL_SAVE}/target',
]:
    os.makedirs(d, exist_ok=True)

print("Paths configured:")
print(f"  Drive code    : {DRIVE_CODE}")
print(f"  Drive saves   : {DRIVE_SAVE}")
print(f"  Local code    : {LOCAL_CODE}")
print(f"  Local saves   : {LOCAL_SAVE}")


---
## Cell 4 — Copy Code from Drive to Local Colab Storage

Local Colab storage (NVMe) is ~10x faster than Drive for imports and file I/O.
We copy your Python source files once per session.

In [ ]:
import shutil
import os

# Verify the Drive code folder has files
drive_files = os.listdir(DRIVE_CODE)
if not drive_files:
    raise FileNotFoundError(
        f"No files found in {DRIVE_CODE}!\n"
        "Please upload all your .py files (train.py, model.py, environment.py, "
        "search.py, utils.py, solve.py, generate_dataset.py) and test_dataset.pt "
        "to MyDrive/MalteseGearCube/code/ on Google Drive, then re-run this cell."
    )

print(f"Found {len(drive_files)} file(s) in Drive code folder. Copying to local...")

for fname in drive_files:
    src = f'{DRIVE_CODE}/{fname}'
    dst = f'{LOCAL_CODE}/{fname}'
    shutil.copy2(src, dst)
    size_kb = os.path.getsize(src) / 1024
    print(f"  Copied: {fname:<35} ({size_kb:>8.1f} KB)")

# Set working directory to local code folder
os.chdir(LOCAL_CODE)
print(f"\nWorking directory: {os.getcwd()}")

# Quick import check
required_files = ['train.py', 'model.py', 'environment.py', 'search.py', 'utils.py']
missing = [f for f in required_files if not os.path.exists(f'{LOCAL_CODE}/{f}')]
if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")
print("All required Python files present.")


---
## Cell 5 — Resume Check: Copy Checkpoint from Drive to Local

If a checkpoint exists on Drive (from a previous session), it is copied to local
so that training resumes from exactly where it left off.
If no checkpoint exists, training starts fresh.

In [ ]:
import os
import shutil
import pickle

DRIVE_CHECKPOINT = f'{DRIVE_SAVE}/training_state.pkl'
LOCAL_CHECKPOINT = f'{LOCAL_SAVE}/training_state.pkl'

if os.path.exists(DRIVE_CHECKPOINT):
    print("Checkpoint found on Drive — copying to local for fast I/O...")
    print("(This may take 1-2 minutes for large checkpoints)\n")

    total_bytes = 0
    for item in sorted(os.listdir(DRIVE_SAVE)):
        src = f'{DRIVE_SAVE}/{item}'
        dst = f'{LOCAL_SAVE}/{item}'
        if os.path.isfile(src):
            shutil.copy2(src, dst)
            size_mb = os.path.getsize(src) / 1e6
            total_bytes += os.path.getsize(src)
            print(f"  Copied: {item:<40} ({size_mb:>8.1f} MB)")
        elif os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            dir_size = sum(
                os.path.getsize(os.path.join(src, f))
                for f in os.listdir(src) if os.path.isfile(os.path.join(src, f))
            )
            total_bytes += dir_size
            print(f"  Copied: {item+'/':<40} ({dir_size/1e6:>8.1f} MB)")

    print(f"\nTotal copied: {total_bytes/1e6:.1f} MB")

    # Display checkpoint state
    with open(LOCAL_CHECKPOINT, 'rb') as f:
        state = pickle.load(f)

    curriculum_names = {0: 'Stage 0 (depth 30)', 1: 'Stage 1 (depth 60)',
                        2: 'Stage 2 (depth 120)', 3: 'Stage 3 (depth 200)',
                        4: 'Stage 4 (depth 500)'}
    stage_idx = state.get('curriculum_stage_idx', 0)

    print("\n" + "="*55)
    print("RESUME STATE")
    print("="*55)
    print(f"  Total gradient steps : {state['itr']:>12,}")
    print(f"  Total update cycles  : {state['update_num']:>12,}")
    print(f"  Curriculum stage     : {curriculum_names.get(stage_idx, stage_idx)}")
    print(f"  Current back_max     : {state['current_back_max']:>12} moves")
    print(f"  Stage update cycle   : {state['stage_update_cycle']:>12}")
    print(f"  All-zeros mode       : {state['use_all_zeros']}")
    print(f"  KL history length    : {len(state.get('kl_history', []))}")
    print("="*55)
    print(f"\nTraining will resume from iteration {state['itr']:,}.")

else:
    print("No checkpoint found on Drive.")
    print("Training will start fresh from scratch (all-zeros initialization).")


---
## Cell 6 — Start Background Drive Sync

This cell launches a background thread that copies the local `saved_models/` folder
to Drive every 5 minutes. This keeps your checkpoint safe even if the session
disconnects between scheduled saves.

**Always run this cell before starting training.**

In [ ]:
import threading
import shutil
import time
import os

SYNC_INTERVAL = 300  # seconds between Drive syncs (5 minutes)

_stop_sync = threading.Event()

def _drive_sync_worker():
    """Background thread: sync local saved_models to Drive every SYNC_INTERVAL seconds."""
    while not _stop_sync.wait(SYNC_INTERVAL):
        try:
            if os.path.exists(f'{LOCAL_SAVE}/training_state.pkl'):
                shutil.copytree(LOCAL_SAVE, DRIVE_SAVE, dirs_exist_ok=True)
                ts = time.strftime('%H:%M:%S')
                print(f"\n[SYNC {ts}] Checkpoint backed up to Drive.", flush=True)
        except Exception as exc:
            print(f"\n[SYNC] Warning — sync failed: {exc}", flush=True)

# Stop any previously running sync thread (in case this cell is re-run)
try:
    _stop_sync.set()
    time.sleep(0.5)
except Exception:
    pass

_stop_sync = threading.Event()
_sync_thread = threading.Thread(target=_drive_sync_worker, daemon=True, name='DriveSync')
_sync_thread.start()

print(f"Background Drive sync started.")
print(f"  Interval : every {SYNC_INTERVAL} s ({SYNC_INTERVAL // 60} min)")
print(f"  Source   : {LOCAL_SAVE}")
print(f"  Target   : {DRIVE_SAVE}")
print()
print("Checkpoints are automatically backed up to Drive while training runs.")


---
## Cell 7 — Start / Resume Training

### Hyperparameters (tuned for A100 40 GB)

| Parameter | Value | Notes |
|---|---|---|
| `states_per_update` | 20,000,000 | Bumped from 15M; fits in ~3 GB as uint8 |
| `TRAIN_BATCH_SIZE` | 45,000 | Auto-detected by `get_hw_profile` |
| `EVAL_BATCH_SIZE` | 45,000 | Auto-detected by `get_hw_profile` |
| `CHILD_EVAL_CHUNK` | 200,000 | Auto-detected by `get_hw_profile` |
| `SCRAMBLE_CHUNK` | 2,000,000 | Auto-detected by `get_hw_profile` |
| `gbfs_num_test` | 5,000 | Doubled from 2500 for more accurate eval |
| All other params | defaults | Unchanged from your original training config |

**Output** streams live into this cell. Training logs are also written to
`saved_models/training_log.txt` (synced to Drive every 5 min).

In [ ]:
import subprocess
import sys
import os

# ── Hyperparameters ───────────────────────────────────────────────────────────
# TRAIN_BATCH_SIZE is set in utils.py → get_hw_profile() → 100,000 for A100 40 GB
# EVAL_BATCH_SIZE  = 45,000  (unchanged — eval doesn't benefit from larger batch)
# CHILD_EVAL_CHUNK = 200,000 (unchanged)
# SCRAMBLE_CHUNK   = 2,000,000 (unchanged)
#
# Expected VRAM with batch=100K: ~28-32 GB (up from 16.5 GB at batch=45K)
# states_per_update bumped to 25M to compensate for fewer gradient steps/cycle:
#   old: 20M / 45K  = 444 steps/epoch
#   new: 25M / 100K = 250 steps/epoch

STATES_PER_UPDATE     = 25_000_000   # 25M states/cycle (~3.6 GB as uint8)
MAX_ITRS              = 1_000_000
LR                    = 0.001
LR_DECAY              = 0.9999998
MAX_DIST              = 500
HIDDEN_DIM            = 7000
RES_DIM               = 2000
NUM_RES_BLOCKS        = 6
CURRICULUM            = ['30', '60', '120', '200', '500']
MAX_INNER_EPOCHS      = 1
EARLY_STOP_PATIENCE   = 2
MIN_IMPROVEMENT       = 0.01
KL_BASE               = 0.05
KL_SCALE_POWER        = 0.5
KL_STAG_PATIENCE      = 6
KL_STAG_MIN_DELTA     = 0.005
MAX_STAGE_CYCLES      = 1000
GBFS_EVAL_FREQ        = 5
GBFS_NUM_TEST         = 5000
GBFS_SOLVE_THRESHOLD  = 75
MIN_SOLVE_DEPTH_FRAC  = 0.60
SIGMA                 = 0.75
GRAD_CLIP             = 1.0

# ── Build command ─────────────────────────────────────────────────────────────
cmd = [
    sys.executable, '-u', 'train.py',
    '--save_dir',                LOCAL_SAVE,
    '--states_per_update',       str(STATES_PER_UPDATE),
    '--max_itrs',                str(MAX_ITRS),
    '--lr',                      str(LR),
    '--lr_decay',                str(LR_DECAY),
    '--max_dist',                str(MAX_DIST),
    '--hidden_dim',              str(HIDDEN_DIM),
    '--res_dim',                 str(RES_DIM),
    '--num_res_blocks',          str(NUM_RES_BLOCKS),
    '--curriculum',              *CURRICULUM,
    '--max_inner_epochs',        str(MAX_INNER_EPOCHS),
    '--early_stop_patience',     str(EARLY_STOP_PATIENCE),
    '--min_improvement',         str(MIN_IMPROVEMENT),
    '--kl_base',                 str(KL_BASE),
    '--kl_scale_power',          str(KL_SCALE_POWER),
    '--kl_stagnation_patience',  str(KL_STAG_PATIENCE),
    '--kl_stagnation_min_delta', str(KL_STAG_MIN_DELTA),
    '--max_stage_cycles',        str(MAX_STAGE_CYCLES),
    '--gbfs_eval_freq',          str(GBFS_EVAL_FREQ),
    '--gbfs_num_test',           str(GBFS_NUM_TEST),
    '--gbfs_solve_threshold',    str(GBFS_SOLVE_THRESHOLD),
    '--min_solve_depth_fraction',str(MIN_SOLVE_DEPTH_FRAC),
    '--sigma',                   str(SIGMA),
    '--grad_clip',               str(GRAD_CLIP),
]

print("Starting training with the following command:")
print(' '.join(cmd))
print()
print(f"Checkpoints saved to : {LOCAL_SAVE}")
print(f"Drive backup         : {DRIVE_SAVE} (every {SYNC_INTERVAL // 60} min)")
print("=" * 60)
print()

# ── Run training — stream output live ─────────────────────────────────────────
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

process = subprocess.Popen(
    cmd,
    cwd=LOCAL_CODE,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

try:
    for line in process.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    process.terminate()
    print("\nTraining interrupted by user.")
finally:
    process.wait()

print()
if process.returncode == 0:
    print("Training completed successfully!")
elif process.returncode == -15:
    print("Training was terminated (SIGTERM).")
else:
    print(f"Training exited with return code {process.returncode}.")

print("\nRun Cell 8 to do a final sync of all files to Drive.")

---
## Cell 8 — Final Sync to Drive

Run this cell after training completes (or before ending your session) to make
sure everything is backed up. The background sync runs every 5 minutes, but
this guarantees the very latest checkpoint is on Drive.

In [ ]:
import shutil
import os
import time

print(f"Syncing {LOCAL_SAVE} → {DRIVE_SAVE} ...")
shutil.copytree(LOCAL_SAVE, DRIVE_SAVE, dirs_exist_ok=True)
print("Sync complete.")

print("\nFiles now on Drive:")
total_bytes = 0
for root, dirs, files in os.walk(DRIVE_SAVE):
    level   = root.replace(DRIVE_SAVE, '').count(os.sep)
    indent  = '  ' * level
    relroot = os.path.basename(root)
    print(f"{indent}{relroot}/")
    for fname in sorted(files):
        fpath   = os.path.join(root, fname)
        size_mb = os.path.getsize(fpath) / 1e6
        total_bytes += os.path.getsize(fpath)
        print(f"{'  ' * (level + 1)}{fname:<40} {size_mb:>8.1f} MB")
print(f"\nTotal on Drive: {total_bytes / 1e9:.2f} GB")
